In [1]:
from ultralytics import YOLO
from manga_ocr import MangaOcr
import cv2
from PIL import Image
from transformers import pipeline
import numpy as np

c:\Users\majd\Desktop\pfe project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
panel_detector = YOLO("models/mosesb.pt")

In [3]:
bubble_detector = YOLO("models/comic-speech-bubble-detector.pt")

In [4]:
ocr = MangaOcr()

2026-02-19 22:02:43.818 | INFO     | manga_ocr.ocr:__init__:16 - Loading OCR model from kha-white/manga-ocr-base
Loading weights: 100%|██████████| 264/264 [00:00<00:00, 272.18it/s, Materializing param=encoder.pooler.dense.weight]                                       
The tied weights mapping and config for this model specifies to tie decoder.bert.embeddings.word_embeddings.weight to decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie decoder.cls.predictions.bias to decoder.cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
MangaOcrModel LOAD REPORT from: kha-white/manga-ocr-base
Key                                  | Status     |  | 
------------------

In [5]:
image_file = "img/005.jpg"

In [6]:
img = cv2.imread(image_file)

In [7]:
panel_results = panel_detector(img, conf=0.2)


0: 640x480 6 Comic Panels, 1910.4ms
Speed: 204.0ms preprocess, 1910.4ms inference, 13.9ms postprocess per image at shape (1, 3, 640, 480)


In [8]:
anotation = panel_results[0].plot()
img_results ="panel_detection.jpg"
cv2.imwrite(img_results,anotation)

True

In [9]:
number = 1
panels = []
for box in panel_results[0].boxes:
    x1, y1, x2, y2 = map(int, box.xyxy[0])
    panels.append({
        "x1": x1, "y1": y1, "x2": x2, "y2": y2,
        "crop": img[y1:y2, x1:x2]
    }
    )
    crop = img[y1:y2, x1:x2]
    cv2.imwrite(f"crop{number}.jpg",crop)
    number +=1

In [10]:
h, w = img.shape[:2]

# ── Helper: Compute IoU between two boxes [x1,y1,x2,y2] ───────────────────────
def compute_iou(boxA, boxB):
    # Intersection coordinates
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])
    
    inter_area = max(0, xB - xA) * max(0, yB - yA)
    
    # Areas
    boxA_area = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
    boxB_area = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])
    
    if boxA_area + boxB_area - inter_area == 0:
        return 0.0
    return inter_area / float(boxA_area + boxB_area - inter_area)

# ── Step 1: Detect panels on full page ───────────────────────────────────────
print("Detecting panels...")
panel_results = panel_detector(img, conf=0.25)
panels = []
for box in panel_results[0].boxes:
    x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
    conf = float(box.conf)
    if conf >= 0.25:
        panels.append({
            "box": [x1, y1, x2, y2],   # for IoU
            "crop": img[y1:y2, x1:x2],
            "conf": conf
        })
print(f"Found {len(panels)} panels")

# ── Step 2: Detect bubbles on FULL page ──────────────────────────────────────
print("Detecting bubbles on full page...")
bubble_results = bubble_detector(img, conf=0.3)
bubbles = []
for box in bubble_results[0].boxes:
    x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
    conf = float(box.conf)
    if conf >= 0.25:
        cx = (x1 + x2) / 2
        cy = (y1 + y2) / 2
        bubble_crop = img[y1:y2, x1:x2]
        bubble_pil = Image.fromarray(cv2.cvtColor(bubble_crop, cv2.COLOR_BGR2RGB))
        
        bubbles.append({
            "box": [x1, y1, x2, y2],
            "cx": cx,
            "cy": cy,
            "crop_pil": bubble_pil,
            "conf": conf
        })
print(f"Found {len(bubbles)} bubbles")

# ── Step 3: Assign bubbles to panels using best IoU ──────────────────────────
panel_to_bubbles = {i: [] for i in range(len(panels))}

for bubble in bubbles:
    best_iou = 0.0
    best_panel_idx = -1
    
    for idx, panel in enumerate(panels):
        iou = compute_iou(bubble["box"], panel["box"])
        if iou > best_iou:
            best_iou = iou
            best_panel_idx = idx
    
    # Fallback: if no good IoU, assign by center point
    if best_iou < 0.1:  # adjust threshold if needed
        bubble_center = (bubble["cx"], bubble["cy"])
        for idx, panel in enumerate(panels):
            bx1, by1, bx2, by2 = panel["box"]
            if bx1 <= bubble["cx"] <= bx2 and by1 <= bubble["cy"] <= by2:
                best_panel_idx = idx
                break
    
    if best_panel_idx != -1:
        panel_to_bubbles[best_panel_idx].append(bubble)
    else:
        print(f"Warning: Bubble {bubble['box']} not assigned to any panel")

# ── Step 4: Sort panels in reading order ─────────────────────────────────────
def get_center(item):
    box = item["box"] if "box" in item else [item["x1"], item["y1"], item["x2"], item["y2"]]
    return ((box[0] + box[2]) / 2, (box[1] + box[3]) / 2)

# Sort: top-to-bottom (cy asc), then right-to-left (-cx desc)
sorted_panel_indices = sorted(
    range(len(panels)),
    key=lambda i: (get_center(panels[i])[1], -get_center(panels[i])[0])
)

print(f"Sorted {len(sorted_panel_indices)} panels in manga reading order")

# ── Step 5: For each sorted panel → sort its bubbles ─────────────────────────
all_ordered_bubbles = []

for panel_idx in sorted_panel_indices:
    panel_bubbles = panel_to_bubbles[panel_idx]
    
    # Sort bubbles inside panel: top-to-bottom, right-to-left
    sorted_bubbles = sorted(
        panel_bubbles,
        key=lambda b: (b["cy"], -b["cx"])
    )
    
    all_ordered_bubbles.extend(sorted_bubbles)

print(f"Final ordered bubbles: {len(all_ordered_bubbles)}")

Detecting panels...

0: 640x480 6 Comic Panels, 2387.7ms
Speed: 11.2ms preprocess, 2387.7ms inference, 1.6ms postprocess per image at shape (1, 3, 640, 480)
Found 6 panels
Detecting bubbles on full page...

0: 1024x736 9 text_bubbles, 2503.7ms
Speed: 16.1ms preprocess, 2503.7ms inference, 34.4ms postprocess per image at shape (1, 3, 1024, 736)
Found 9 bubbles
Sorted 6 panels in manga reading order
Final ordered bubbles: 9


In [11]:
from transformers import MarianMTModel, MarianTokenizer

# Load the Japanese → Arabic model
model_name = "Helsinki-NLP/opus-mt-ja-ar"
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)

c:\Users\majd\Desktop\pfe project\.venv\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\majd\.cache\huggingface\hub\models--Helsinki-NLP--opus-mt-ja-ar. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\majd\Desktop\pfe project\.venv\Lib\site-packages\transformers\models\marian\tokenization_maria

In [12]:
def translate_ja_to_ar(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        src_lang="jpn_Jpan"
    )

    arb_id = tokenizer.convert_tokens_to_ids("fra_Latn")

    generated_tokens = model.generate(
        **inputs,
        forced_bos_token_id=arb_id,
        max_length=256
    )

    return tokenizer.decode(generated_tokens[0], skip_special_tokens=True)

print(translate_ja_to_ar("私は学生です。"))

c.arabic c.arabicrlm; أنا طالب. /c.arabic c.arabicrlm;


In [ ]:
bubble_number = 1
for bubble in all_ordered_bubbles:
    japanese = ocr(bubble["crop_pil"]).strip()
    print(f"Bubble {bubble_number}:")
    print(f"Japanese: {japanese}")
    print("─" * 50)
    bubble_number += 1
    try:
    #  Tokenize input (MarianMT expects the text directly)
     inputs = tokenizer(japanese, return_tensors="pt", padding=True)
    
    #  Generate translation
     translated_ids = model.generate(
        **inputs,
        max_length=200,
        num_beams=4,          # optional: improves quality a bit
        early_stopping=True
    )
    
     arabic = tokenizer.decode(translated_ids[0], skip_special_tokens=True).strip()
     print(arabic)
    except Exception as e:
     print(f"Bubble {bubble_number}: Translation failed → {e}")
     arabic = "[translation error]"
    

Bubble 1:
Japanese: ちっくしょ〜
──────────────────────────────────────────────────
تباً!
Bubble 2:
Japanese: 手伝わんなら推薦はやらん
──────────────────────────────────────────────────
إذا كنت تريد مساعدتي، لن أرشحك.
Bubble 3:
Japanese: 脅迫じゃねーかあのパワハラ教師！！
──────────────────────────────────────────────────
لا تبتزني أيها المُعلّم العاهر
Bubble 4:
Japanese: なぜ俺がこんなことを．．．
──────────────────────────────────────────────────
لمَ قد أفعل هذا؟
Bubble 5:
Japanese: ．．．天文部
──────────────────────────────────────────────────
مَلَك
Bubble 6:
Japanese: ここか
──────────────────────────────────────────────────
هنا؟
Bubble 7:
Japanese: うぉぉぉおぉおおおぉ！？
──────────────────────────────────────────────────
يا إلهي!
Bubble 8:
Japanese: 失礼しまーす
──────────────────────────────────────────────────
شكراً لكِ
Bubble 9:
Japanese: ん？
──────────────────────────────────────────────────
ماذا؟
